# Introduction to integer programming
*Algorithms for Optimization, Master Program in Finance, University of Pavia. Notebook of Lectures 4 and 5.*

| Section | Topic | Lecture |
|---|---|---|
| 0 | Setup | |
| 1 | From enumeration to integer programming | 4 |
| 2 | Capital budgeting: rounding is not enough | 4 |
| 3 | Investment in bond bundles | 4 |
| 4 | Exercise: oil exploration | 4 |
| 5 | Exercise: rules for the 25 projects | 4 |
| 6 | Branch-and-bound: watching the solver | 5 |
| 7 | Formulations matter: big-M | 5 |
| 8 | Exercise: combinatorial auctions | 5 |
| 9 | Exercise: portfolio rebalancing | 5 |
| 10 | At home, for fun: Sudoku | |

Exercises are marked with ✏️.

## 0. Setup

In [ ]:
!pip install gurobipy

In [ ]:
import time
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/AmbrogioMB/AlgOpt/main/data/"

## 1. From enumeration to integer programming
In the first lecture of the course Prof. Duma solved the capital allocation problem (budget 100, projects A, B, C) by **enumeration**: try all the $2^n$ selections and keep the best feasible one. Here is the same algorithm, written as a function.

In [ ]:
def capital_allocation_enumeration(c, p, B):
    n = len(c)
    P_best = 0
    x_best = [0] * n
    k = 0
    while k < 2**n:
        x = [(k >> i) & 1 for i in range(n)]   # the n-bit representation of k
        C = sum(c[i] * x[i] for i in range(n))
        P = sum(p[i] * x[i] for i in range(n))
        if C <= B:
            if P > P_best:
                P_best = P
                x_best = x.copy()
        k += 1
    return x_best, P_best


print(capital_allocation_enumeration([60, 50, 40], [30, 25, 22], 100))

Now a bigger instance: 25 projects in the file `projects.csv` (investment and expected profit in thousands of euros), and a budget of 500.

In [ ]:
projects = pd.read_csv(DATA_URL + "projects.csv")
B = 500
projects.head(10)

How long does the enumeration take? We time it on the first $n$ projects, for increasing $n$.

In [ ]:
cost = list(projects["investment"])
profit = list(projects["expected_profit"])

sizes, times = [], []
for n in [8, 10, 12, 14, 16]:
    start = time.perf_counter()
    capital_allocation_enumeration(cost[:n], profit[:n], B)
    sizes.append(n)
    times.append(time.perf_counter() - start)
    print(f"n = {n:2d}: {2**n:7d} selections, {times[-1]:.3f} seconds")

per_selection = times[-1] / 2**sizes[-1]
for n in [25, 50]:
    seconds = per_selection * 2**n
    print(f"prediction for n = {n}: {seconds:.3g} seconds = {seconds / 3600 / 24 / 365:.3g} years")

plt.semilogy(sizes, times, "o-")
plt.xlabel("number of projects n")
plt.ylabel("time (seconds, log scale)")
plt.title("Enumeration: the time doubles with every new project")
plt.grid(True)
plt.show()

The same problem as an **integer program**: $x_i \in \{0, 1\}$,
$$
\max \sum_i p_i x_i \quad \text{s.t.} \quad \sum_i c_i x_i \leq B, \quad x_i \in \{0, 1\}.
$$
The only difference with an LP is `vtype=GRB.BINARY`. Look at the log: how many nodes does Gurobi explore? (Lecture 5 explains what a node is.)

In [ ]:
def capital_allocation_ilp(c, p, B, verbose=True):
    n = len(c)
    m = gp.Model("Capital allocation")
    m.Params.OutputFlag = 1 if verbose else 0
    x = m.addVars(n, vtype=GRB.BINARY, name="x")
    m.addConstr(gp.quicksum(c[i] * x[i] for i in range(n)) <= B, name="budget")
    m.setObjective(gp.quicksum(p[i] * x[i] for i in range(n)), GRB.MAXIMIZE)
    m.optimize()
    return [round(x[i].X) for i in range(n)], m.ObjVal, m


x_ilp, value, m = capital_allocation_ilp(cost, profit, B)
selected = [projects["project"][i] for i in range(len(cost)) if x_ilp[i] == 1]
print(f"\nSelected projects: {selected}")
print(f"Expected profit: {value:.1f}, time: {m.Runtime:.3f} seconds, nodes: {m.NodeCount:.0f}")

### Exercise 1.1 ✏️
An implementation must be **validated** (first part of the course). Check that the enumeration and Gurobi give the same optimal profit on the first 16 projects.

In [ ]:
# YOUR CODE HERE

## 2. Capital budgeting: rounding is not enough
Suppose we wish to invest €19,000. Investment 1 requires €6,700 and has a net present value of €8,000; investment 2 requires €10,000 and has a value of €11,000; investment 3 requires €5,500 and has a value of €6,000; investment 4 requires €3,400 and has a value of €4,000. Each project is a "take it or leave it" opportunity. We use a binary variable $x_j$ for each investment:
$$
\max \; 8x_1 + 11x_2 + 6x_3 + 4x_4 \quad \text{s.t.} \quad 6.7x_1 + 10x_2 + 5.5x_3 + 3.4x_4 \leq 19, \quad x_j \in \{0, 1\}.
$$
First we solve the **linear relaxation** ($0 \leq x_j \leq 1$).

In [ ]:
investments = [1, 2, 3, 4]
inv_cost = {1: 6.7, 2: 10, 3: 5.5, 4: 3.4}
npv = {1: 8, 2: 11, 3: 6, 4: 4}

relax = gp.Model("Capital budgeting, LP relaxation")
relax.Params.OutputFlag = 0
xr = relax.addVars(investments, ub=1, name="x")
relax.addConstr(gp.quicksum(inv_cost[j] * xr[j] for j in investments) <= 19, name="budget")
relax.setObjective(gp.quicksum(npv[j] * xr[j] for j in investments), GRB.MAXIMIZE)
relax.optimize()
print("LP relaxation:", {j: round(xr[j].X, 3) for j in investments}, "value", round(relax.ObjVal, 3))

x_round = {j: int(xr[j].X) for j in investments}  # round down
print("Rounded down: ", x_round, "value", sum(npv[j] * x_round[j] for j in investments))

In [ ]:
cb = gp.Model("Capital budgeting")
xb = cb.addVars(investments, vtype=GRB.BINARY, name="x")
cb.addConstr(gp.quicksum(inv_cost[j] * xb[j] for j in investments) <= 19, name="budget")
cb.setObjective(gp.quicksum(npv[j] * xb[j] for j in investments), GRB.MAXIMIZE)
cb.optimize()
print("\nOptimal integer solution:", {j: round(xb[j].X) for j in investments}, "value", cb.ObjVal)

Rounding gives 12, the optimum is 21, and the LP value 21.79 is an **upper bound** on the integer optimum.

### 2.1 Logical constraints
1. We can make at most two investments: $x_1 + x_2 + x_3 + x_4 \leq 2$.
2. If investment 2 is made, then investment 4 must also be made: $x_2 \leq x_4$.
3. If investment 1 is made, then investment 3 cannot be made: $x_1 + x_3 \leq 1$.

In [ ]:
cb.addConstr(xb.sum() <= 2, name="at_most_two")
cb.addConstr(xb[2] <= xb[4], name="2_implies_4")
cb.addConstr(xb[1] + xb[3] <= 1, name="1_excludes_3")
cb.Params.OutputFlag = 0
cb.optimize()
print("With logical constraints:", {j: round(xb[j].X) for j in investments}, "value", cb.ObjVal)

## 3. Investment in bond bundles
An investor has €50,000 to invest in three types of corporate bonds, sold only in bundles. The investor can buy several bundles of each type, but only whole bundles (thousands of euros):

| Bond | Cost per bundle | Expected return | Max bundles |
|---|---|---|---|
| A | 6.0 | 0.72 | 5 |
| B | 8.5 | 1.00 | 4 |
| C | 4.0 | 0.44 | 6 |

The variables are **general integers** (`vtype=GRB.INTEGER`), with upper bounds.

In [ ]:
bonds = ["A", "B", "C"]
bundle_cost = {"A": 6.0, "B": 8.5, "C": 4.0}
bundle_ret = {"A": 0.72, "B": 1.00, "C": 0.44}
max_bundles = {"A": 5, "B": 4, "C": 6}

bb = gp.Model("Bond bundles")
bb.Params.OutputFlag = 0
nb = bb.addVars(bonds, vtype=GRB.INTEGER, ub=max_bundles, name="bundles")
bb.addConstr(gp.quicksum(bundle_cost[k] * nb[k] for k in bonds) <= 50, name="budget")
bb.setObjective(gp.quicksum(bundle_ret[k] * nb[k] for k in bonds), GRB.MAXIMIZE)
bb.optimize()
print({k: round(nb[k].X) for k in bonds}, "expected return", round(bb.ObjVal, 2))

## 4. Exercise ✏️: oil exploration (Exercise 1 of Lecture 4)
As the leader of an oil exploration drilling venture, you must determine the best selection of 5 out of 10 possible sites $s_1, \dots, s_{10}$, with expected profits $p_1, \dots, p_{10}$ (below).
1. If site $s_2$ is explored, then site $s_3$ must also be explored.
2. Exploring sites $s_1$ and $s_7$ will prevent you from exploring site $s_8$.
3. Exploring sites $s_3$ or $s_4$ will prevent you from exploring site $s_5$.

Write and solve the model. Careful with the indices: in Python the site $s_1$ is `x[0]`.

In [ ]:
p_oil = np.random.default_rng(1).integers(1, 20, 10)
print("Profits:", p_oil)

In [ ]:
# YOUR CODE HERE

## 5. Exercise ✏️: rules for the 25 projects
Go back to the 25 projects of Section 1 (budget 500) and add, one at a time, the following rules. After each rule, solve again and look at how the selection and the expected profit change.
1. At most 1 project in the Energy sector.
2. Project P25 can be selected only if project P05 is selected (P05 builds the platform that P25 uses).
3. Projects P04 and P14 are mutually exclusive.
4. The average risk score of the selected projects must be at most 6. *Hint:* $\frac{\sum_j r_j x_j}{\sum_j x_j} \leq 6$ is not linear, but it is equivalent to $\sum_j (r_j - 6) x_j \leq 0$.
5. Each sector in which we invest has a fixed management cost of 10, subtracted from the expected profit. *Hint:* one binary variable $y_s$ per sector, with $x_j \leq y_s$ for every project $j$ of sector $s$.

In [ ]:
names = list(projects["project"])
inv = dict(zip(names, projects["investment"]))
prof = dict(zip(names, projects["expected_profit"]))
risk = dict(zip(names, projects["risk_score"]))
proj_sector = dict(zip(names, projects["sector"]))

rules = gp.Model("Projects with rules")
rules.Params.OutputFlag = 0
xp = rules.addVars(names, vtype=GRB.BINARY, name="x")
rules.addConstr(gp.quicksum(inv[j] * xp[j] for j in names) <= B, name="budget")
rules.setObjective(gp.quicksum(prof[j] * xp[j] for j in names), GRB.MAXIMIZE)


def report(model, x):
    chosen = [j for j in names if x[j].X > 0.5]
    print(f"profit {model.ObjVal:.1f}, invested {sum(inv[j] for j in chosen)}, projects {chosen}")


rules.optimize()
report(rules, xp)

In [ ]:
# YOUR CODE HERE

## 6. Branch-and-bound: watching the solver
We do not implement branch-and-bound: Gurobi does it for us. Here we look at what it does, and we draw a tree on paper.

### 6.1 The example of the slides (Exercise 1 of Lecture 5)
$$
\max \; 5.5x_1 + 2.1x_2 \quad \text{s.t.} \quad -x_1 + x_2 \leq 2, \quad 8x_1 + 2x_2 \leq 17, \quad x_1, x_2 \geq 0 \text{ integer}.
$$
Solve it and look at the log. How many nodes does Gurobi explore? In the slides we needed 5. Why the difference? *Hint:* look at the line `Root relaxation`, and recall what presolve does.

In [ ]:
ex_bb = gp.Model("B&B example")
x1 = ex_bb.addVar(vtype=GRB.INTEGER, name="x1")
x2 = ex_bb.addVar(vtype=GRB.INTEGER, name="x2")
ex_bb.setObjective(5.5 * x1 + 2.1 * x2, GRB.MAXIMIZE)
ex_bb.addConstr(-x1 + x2 <= 2)
ex_bb.addConstr(8 * x1 + 2 * x2 <= 17)
ex_bb.optimize()
print(f"\nx1 = {x1.X}, x2 = {x2.X}, value {ex_bb.ObjVal}, nodes explored {ex_bb.NodeCount:.0f}")

### 6.2 Exercise ✏️: draw the tree (Exercise 2 of Lecture 5)
Draw on paper the branch-and-bound tree of the capital budgeting problem of Section 2 (€19,000, four investments), branching on the fractional variable. The root is the LP relaxation: $x = (1, 0.89, 0, 1)$, $z = 21.79$.

To find the LP solution of each node of your tree, use the function `node_lp` below: you only have to **call** it, with the variables fixed by the branching. For example `node_lp({2: 0})` solves the LP relaxation with $x_2 = 0$, and `node_lp({2: 1, 4: 0})` the one with $x_2 = 1$ and $x_4 = 0$. For each node, decide: branch again, or prune (by infeasibility, integrality or bound)?

In [ ]:
def node_lp(fixed):
    """LP relaxation of the capital budgeting problem with the variables in `fixed` fixed to 0 or 1."""
    m = gp.Model()
    m.Params.OutputFlag = 0
    x = m.addVars(investments, lb=0, ub=1, name="x")
    for j, value in fixed.items():
        x[j].LB = value
        x[j].UB = value
    m.addConstr(gp.quicksum(inv_cost[j] * x[j] for j in investments) <= 19)
    m.setObjective(gp.quicksum(npv[j] * x[j] for j in investments), GRB.MAXIMIZE)
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        print(f"fixed {fixed}: infeasible")
    else:
        solution = {j: round(x[j].X, 3) for j in investments}
        print(f"fixed {fixed}: x = {solution}, z = {m.ObjVal:.3f}")


node_lp({})  # the root

In [ ]:
# YOUR CODE HERE: call node_lp for the nodes of your tree

### 6.3 Branch-and-bound on the 25 projects
Enumeration needs $2^{25} \approx 33$ million selections. How many nodes does Gurobi need? First with its default settings, then with presolve, cutting planes and heuristics switched off, i.e., (almost) the plain branch-and-bound of the slides.

In [ ]:
for plain in [False, True]:
    m = gp.Model("Capital allocation")
    m.Params.OutputFlag = 0
    if plain:
        m.Params.Presolve, m.Params.Cuts, m.Params.Heuristics = 0, 0, 0
    x = m.addVars(len(cost), vtype=GRB.BINARY, name="x")
    m.addConstr(gp.quicksum(cost[i] * x[i] for i in range(len(cost))) <= B)
    m.setObjective(gp.quicksum(profit[i] * x[i] for i in range(len(cost))), GRB.MAXIMIZE)
    m.optimize()
    label = "plain branch-and-bound" if plain else "default settings     "
    print(f"{label}: profit {m.ObjVal:.1f}, nodes {m.NodeCount:.0f}, time {m.Runtime:.3f} s")
print(f"enumeration: {2**len(cost)} selections")

With the default settings Gurobi closes the gap at the root node (0 nodes explored after the root): presolve, cuts and heuristics are very effective. Even plain branch-and-bound needs only about a hundred nodes, against 33 million selections: the LP bounds prune almost the whole tree.

## 7. Formulations matter: big-M (Exercise 3 of Lecture 5)
We invest in 60 assets, at most 10% in each, at most 25% in each of 6 sectors, at most 12 assets. Buying asset $j$ costs a fixed fee $f_j$ (a fraction of the wealth), modeled with a binary variable $\delta_j$ and a big-M constraint $w_j \leq M \delta_j$. If we buy an asset, we buy at least 2%.
$$
\max \sum_j \mu_j w_j - \sum_j f_j \delta_j \quad \text{s.t.} \quad \sum_j w_j = 1, \quad 0.02\, \delta_j \leq w_j \leq M \delta_j, \quad w_j \leq 0.1, \quad \sum_j \delta_j \leq 12, \quad \text{sector limits}.
$$
Since $w_j \leq 0.1$ anyway, every $M \geq 0.1$ gives a correct model. But are all these models equally good? To see the effect on branch-and-bound we switch off presolve, cuts and heuristics.

In [ ]:
rng = np.random.default_rng(7)
n_assets = 60
mu_a = rng.uniform(0.02, 0.15, n_assets)
fee = rng.uniform(0.002, 0.02, n_assets)
sec = rng.integers(0, 6, n_assets)


def fee_model(M, relax=False, plain=True):
    m = gp.Model("Fixed fees")
    m.Params.OutputFlag = 0
    if plain:  # switch off the tricks of Gurobi to see the pure branch-and-bound
        m.Params.Presolve, m.Params.Cuts, m.Params.Heuristics = 0, 0, 0
    w = m.addMVar(n_assets, ub=0.10, name="w")
    d = m.addMVar(n_assets, ub=1, vtype=GRB.CONTINUOUS if relax else GRB.BINARY, name="delta")
    m.setObjective(mu_a @ w - fee @ d, GRB.MAXIMIZE)
    m.addConstr(w.sum() == 1)
    m.addConstr(w <= M * d)
    m.addConstr(w >= 0.02 * d)
    m.addConstr(d.sum() <= 12)
    for s in range(6):
        m.addConstr(w[sec == s].sum() <= 0.25)
    m.optimize()
    return m


for M in [0.1, 1, 100]:
    lp, ip = fee_model(M, relax=True), fee_model(M)
    print(f"M = {M:5}: LP relaxation {lp.ObjVal:.4f}, optimum {ip.ObjVal:.4f}, "
          f"nodes {ip.NodeCount:6.0f}, time {ip.Runtime:.2f} s")

### Exercise 7.1 ✏️
1. Why does the LP relaxation get weaker (larger) when $M$ grows? *Hint:* in the relaxation $\delta_j$ can be fractional; what is the smallest $\delta_j$ allowed for a given $w_j$, and how much fee does it pay?
2. Solve the three models again with the default parameters of Gurobi (`plain=False`). What do you observe? Why?

In [ ]:
# YOUR CODE HERE

## 8. Exercise ✏️: combinatorial auctions
In many auctions, the value that a bidder has for a set of items is not the sum of the values of the individual items (equity trading, electricity markets, pollution rights, airport landing slots). Combinatorial auctions allow bids on **sets** of items.

Let $M = \{1, \dots, m\}$ be the items for sale. A bid is a pair $B_j = (S_j, p_j)$ where $S_j \subseteq M$ is a nonempty set of items and $p_j$ the price offered. Given bids $B_1, \dots, B_n$, which bids should win to maximize the revenue? With $x_j = 1$ if bid $B_j$ wins:
$$
\max \sum_{j=1}^n p_j x_j \quad \text{s.t.} \quad \sum_{j \, : \, i \in S_j} x_j \leq 1, \; i = 1, \dots, m, \qquad x_j \in \{0, 1\}.
$$
Each item is sold at most once.

### Exercise 8.1 ✏️
Solve the problem with the bids below.

In [ ]:
bids = [({1}, 6), ({2}, 3), ({3, 4}, 12), ({1, 3}, 12), ({2, 4}, 8), ({1, 3, 4}, 16)]
items = {1, 2, 3, 4}

In [ ]:
# YOUR CODE HERE

### 8.2 Multiple units
Sometimes there are $u_i$ identical units of each item $i$. A bid is $B_j = (\lambda^j_1, \dots, \lambda^j_m; p_j)$, where $\lambda^j_i$ is the number of units of item $i$ requested. The constraints become
$$
\sum_{j=1}^n \lambda^j_i x_j \leq u_i, \quad i = 1, \dots, m.
$$

### Exercise 8.3 ✏️
Solve the multiple-unit auction with the (random) data below: `units[i]` is $u_i$, `L[j, i]` is $\lambda^j_i$, `price[j]` is $p_j$.

In [ ]:
rng = np.random.default_rng(0)
m_items, n_bids = 5, 10
units = rng.integers(5, 16, m_items)
L = rng.integers(0, 4, size=(n_bids, m_items))
price = rng.integers(10, 101, n_bids)
print("units:", units)
print("prices:", price)

In [ ]:
# YOUR CODE HERE

## 9. Exercise ✏️: portfolio rebalancing
You own a portfolio of eight stocks. Using the Markowitz model (Lecture 6), you computed the optimal mean-variance (M/V) portfolio:

| Stock | A | B | C | D | E | F | G | H |
|---|---|---|---|---|---|---|---|---|
| Your portfolio | 0.12 | 0.15 | 0.13 | 0.10 | 0.20 | 0.10 | 0.12 | 0.08 |
| M/V portfolio | 0.02 | 0.05 | 0.25 | 0.06 | 0.18 | 0.10 | 0.22 | 0.12 |

You want to rebalance your portfolio to be closer to the M/V one, but to limit transaction costs you trade **at most three stocks**. Let $x_i$ be the new weight of stock $i$. Minimize
$$
|x_1 - 0.02| + |x_2 - 0.05| + \cdots + |x_8 - 0.12|.
$$
Formulate it as a MILP: you need new continuous variables to linearize the absolute values (Lecture 4) and binary variables to count the stocks that are traded (big-M: what is a valid $M$?). The new weights must sum to 1.

In [ ]:
stocks = ["A", "B", "C", "D", "E", "F", "G", "H"]
current = dict(zip(stocks, [0.12, 0.15, 0.13, 0.10, 0.20, 0.10, 0.12, 0.08]))
target = dict(zip(stocks, [0.02, 0.05, 0.25, 0.06, 0.18, 0.10, 0.22, 0.12]))

In [ ]:
# YOUR CODE HERE

## 10. At home, for fun: Sudoku
The grid must be filled with the numbers $1, \dots, 9$ so that each number appears once in each row, once in each column and once in each of the nine $3 \times 3$ blocks. We use binary variables $x_{ijk} = 1$ if number $k$ is in cell $(i, j)$. This is a **feasibility problem**: no objective.

The model below is incomplete: it only says that each cell contains exactly one number, and fixes the given numbers. ✏️ Add the constraints for rows, columns and blocks.

In [ ]:
puzzle = [
    [8, 0, 0, 0, 2, 6, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 7, 0, 4],
    [0, 0, 0, 7, 0, 0, 0, 0, 5],
    [0, 0, 0, 1, 0, 0, 0, 3, 6],
    [0, 1, 0, 0, 8, 0, 0, 4, 0],
    [9, 8, 0, 0, 0, 3, 0, 0, 0],
    [3, 0, 0, 0, 0, 1, 0, 0, 0],
    [7, 0, 5, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 2, 5, 0, 0, 0, 8],
]

sudoku = gp.Model("Sudoku")
xs = sudoku.addVars(9, 9, 9, vtype=GRB.BINARY, name="x")  # xs[i, j, k] = 1 if number k+1 is in cell (i, j)

for i in range(9):
    for j in range(9):
        sudoku.addConstr(gp.quicksum(xs[i, j, k] for k in range(9)) == 1)  # one number per cell
        if puzzle[i][j] > 0:
            sudoku.addConstr(xs[i, j, puzzle[i][j] - 1] == 1)              # given numbers

In [ ]:
# YOUR CODE HERE: rows, columns, blocks

In [ ]:
sudoku.Params.OutputFlag = 0
sudoku.optimize()
if sudoku.Status == GRB.OPTIMAL:
    for i in range(9):
        row = ""
        for j in range(9):
            row += str([k + 1 for k in range(9) if xs[i, j, k].X > 0.5][0]) + " "
            if j in (2, 5):
                row += "| "
        print(row)
        if i in (2, 5):
            print("-" * 21)
else:
    print("No solution found.")

### 10.1 Is the solution unique?
To check whether a binary problem has another solution, we add a constraint that **excludes** the solution $z^0$ just found and solve again. If $J$ is the set of variables equal to 1 in $z^0$ and $I$ the set of those equal to 0, the constraint
$$
\sum_{i \in I} z_i + \sum_{j \in J} (1 - z_j) \geq 1
$$
says that at least one variable must change. If the new problem is infeasible, the solution was unique.

In [ ]:
solutions = 0
while True:
    sudoku.optimize()
    if sudoku.Status != GRB.OPTIMAL:
        break
    solutions += 1
    ones = [key for key in xs if xs[key].X > 0.5]
    zeros = [key for key in xs if xs[key].X <= 0.5]
    sudoku.addConstr(gp.quicksum(xs[key] for key in zeros) + gp.quicksum(1 - xs[key] for key in ones) >= 1)
    if solutions >= 5:
        break
print(f"Number of solutions found: {solutions}")